In [ ]:
import gradio as gr
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
import time
import os

from PIL import Image

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.applications import (
    EfficientNetB0,
    EfficientNetB3,
    ResNet50
)

from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

from tensorflow.keras.preprocessing.image import (
    load_img,
    img_to_array
)

from sklearn.metrics.pairwise import cosine_similarity


# ======================================================
# 1. LOAD DATASET
# ======================================================

DATASET_PATH = (
    "/kaggle/input/datasets/ziadabdullah000/data-food/final_food_data/final_refined_food_dataset.csv"
)

IMAGES_FOLDER = (
    "/kaggle/input/datasets/ziadabdullah000/data-food/final_food_data/all_images"
)


df = pd.read_csv(DATASET_PATH)


df["image_path"] = df["image_path"].apply(
    lambda x: os.path.join(
        IMAGES_FOLDER,
        os.path.basename(x)
    )
)

print("DATASET LOADED ✅")


# ======================================================
# 2. LOAD MODELS
# ======================================================

MODELS = {

    "EfficientNetB0 + BiLSTM": {

        "model_path": "/kaggle/input/models/ziadabdullah000/effb0/keras/default/1/final_model.keras",

        "tokenizer_path": "/kaggle/input/models/ziadabdullah000/effb0/keras/default/1/tokenizer.pkl",

        "features_path": "/kaggle/input/models/ziadabdullah000/effb0/keras/default/1/features.pkl",

        "cnn": "b0",

        "feature_dim": 1280,

        "image_size": (224, 224)
    },


    "ResNet50 + Attention BiLSTM": {

        "model_path": "/kaggle/input/models/ziadabdullah000/resnet50/keras/default/1/best_model.keras",

        "tokenizer_path": "/kaggle/input/models/ziadabdullah000/resnet50/keras/default/1/tokenizer.pkl",

        "features_path": "/kaggle/input/models/ziadabdullah000/resnet50/keras/default/1/featuresRESNET50.pkl",

        "cnn": "resnet",

        "feature_dim": 2048,

        "image_size": (224, 224)
    },


    "EfficientNetB3 + BiGRU + MHA": {

        "model_path": "/kaggle/input/models/ziadabdullah000/efficentnetb3/keras/default/1/best_model.keras",

        "tokenizer_path": "/kaggle/input/models/ziadabdullah000/efficentnetb3/keras/default/1/tokenizer.pkl",

        "features_path": "/kaggle/input/models/ziadabdullah000/efficentnetb3/keras/default/1/featuresB3.pkl",

        "cnn": "b3",

        "feature_dim": 1536,

        "image_size": (300, 300)
    }
}


loaded_models = {}

for model_name, config in MODELS.items():

    print(f"LOADING {model_name}...")

    model = load_model(config["model_path"])

    with open(config["tokenizer_path"], "rb") as f:
        tokenizer = pickle.load(f)

    with open(config["features_path"], "rb") as f:
        features = pickle.load(f)

    loaded_models[model_name] = {
        "model": model,
        "tokenizer": tokenizer,
        "features": features,
        "config": config
    }

print("\nALL MODELS LOADED ✅")


# ======================================================
# 3. LOAD CNN BACKBONES
# ======================================================

cnn_b0 = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    pooling='avg'
)

cnn_resnet = ResNet50(
    weights='imagenet',
    include_top=False,
    pooling='avg'
)

cnn_b3 = EfficientNetB3(
    weights='imagenet',
    include_top=False,
    pooling='avg'
)

print("CNN BACKBONES READY ✅")


# ======================================================
# 4. FEATURE EXTRACTION
# ======================================================


def extract_features(img_path, cnn_type, image_size):

    img = load_img(
        img_path,
        target_size=image_size
    )

    img = img_to_array(img)

    img = np.expand_dims(img, axis=0)

    if cnn_type == "resnet":

        img = resnet_preprocess(img)

        features = cnn_resnet.predict(
            img,
            verbose=0
        )

    elif cnn_type == "b3":

        img = efficientnet_preprocess(img)

        features = cnn_b3.predict(
            img,
            verbose=0
        )

    else:

        img = efficientnet_preprocess(img)

        features = cnn_b0.predict(
            img,
            verbose=0
        )

    return features.flatten()


# ======================================================
# 5. INDEX TO WORD
# ======================================================


def build_index_word(tokenizer):

    return {
        index: word
        for word, index in tokenizer.word_index.items()
    }


# ======================================================
# 6. GENERATE CAPTION
# ======================================================


def generate_caption(
    model,
    tokenizer,
    photo,
    max_length,
    feature_dim
):

    index_to_word = build_index_word(tokenizer)

    in_text = "<start>"

    used_words = []

    for i in range(max_length):

        sequence = tokenizer.texts_to_sequences(
            [in_text]
        )[0]

        sequence = pad_sequences(
            [sequence],
            maxlen=max_length
        )

        yhat = model.predict(
            [
                photo.reshape(1, feature_dim),
                sequence
            ],
            verbose=0
        )

        yhat_probs = yhat[0]

        temperature = 0.7

        yhat_probs = np.log(
            yhat_probs + 1e-10
        ) / temperature

        exp_preds = np.exp(yhat_probs)

        yhat_probs = exp_preds / np.sum(exp_preds)

        top_k = 5

        top_indices = np.argsort(
            yhat_probs
        )[-top_k:]

        top_probs = yhat_probs[top_indices]

        top_probs = top_probs / np.sum(top_probs)

        yhat = np.random.choice(
            top_indices,
            p=top_probs
        )

        word = index_to_word.get(yhat)

        if word is None:
            break

        if used_words.count(word) >= 2:
            break

        used_words.append(word)

        in_text += ' ' + word

        if word == '<end>':
            break

    return in_text


# ======================================================
# 7. CLEAN CAPTION
# ======================================================


def clean_caption(caption):

    caption = caption.replace('<start>', '')

    caption = caption.replace('<end>', '')

    caption = caption.lower()

    stop_words = {
        "tablespoon",
        "tablespoons",
        "teaspoon",
        "teaspoons",
        "cup",
        "cups",
        "divided",
        "chopped",
        "sliced",
        "freshly",
        "finely"
    }

    words = caption.split()

    cleaned_words = []

    for word in words:

        if word in stop_words:
            continue

        if len(cleaned_words) > 0:

            if word == cleaned_words[-1]:
                continue

        cleaned_words.append(word)

    final_words = []

    word_counts = {}

    for word in cleaned_words:

        if word not in word_counts:
            word_counts[word] = 0

        if word_counts[word] < 2:

            final_words.append(word)

            word_counts[word] += 1

    final_words = final_words[:18]

    caption = " ".join(final_words)

    caption = caption.capitalize()

    if not caption.endswith("."):
        caption += "."

    return caption


# ======================================================
# 8. RECOMMEND DISHES
# ======================================================


def recommend_dishes(
    query_features,
    features_dict,
    top_n=5
):

    image_names = list(features_dict.keys())

    feature_vectors = np.array(
        list(features_dict.values())
    )

    similarities = cosine_similarity(
        [query_features],
        feature_vectors
    )[0]

    top_indices = similarities.argsort(
    )[-top_n:][::-1]

    recommendations = []

    used_titles = set()

    for idx in top_indices:

        image_name = image_names[idx]

        matched_row = df[
            df['image_path'].str.contains(
                image_name
            )
        ]

        if len(matched_row) > 0:

            title = matched_row.iloc[0]['title']

            if title not in used_titles:

                recommendations.append(title)

                used_titles.add(title)

    return recommendations


# ======================================================
# 9. PREDICTION FUNCTION
# ======================================================


def run_all_models(image):

    image_path = "/kaggle/working/temp_image.jpg"

    image.save(image_path)

    outputs = []

    for model_name, bundle in loaded_models.items():

        model = bundle["model"]

        tokenizer = bundle["tokenizer"]

        features_dict = bundle["features"]

        config = bundle["config"]

        start = time.time()

        query_features = extract_features(
            image_path,
            config["cnn"],
            config["image_size"]
        )

        raw_caption = generate_caption(
            model,
            tokenizer,
            query_features,
            max_length=20,
            feature_dim=config["feature_dim"]
        )

        caption = clean_caption(raw_caption)

        recommendations = recommend_dishes(
            query_features,
            features_dict,
            top_n=5
        )

        end = time.time()

        inference_time = round(end - start, 2)

        result = f"""
### {model_name}

⏱️ Inference Time: {inference_time} sec

🍽️ Caption:
{caption}

🔥 Recommended Dishes:
1. {recommendations[0] if len(recommendations) > 0 else 'N/A'}
2. {recommendations[1] if len(recommendations) > 1 else 'N/A'}
3. {recommendations[2] if len(recommendations) > 2 else 'N/A'}
4. {recommendations[3] if len(recommendations) > 3 else 'N/A'}
5. {recommendations[4] if len(recommendations) > 4 else 'N/A'}
"""

        outputs.append(result)

    return outputs


# ======================================================
# 10. GRADIO UI
# ======================================================

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
# 🍽️ Multi-Model Food Captioning Studio

Compare 3 Deep Learning Models:

- EfficientNetB0 + BiLSTM
- ResNet50 + Attention BiLSTM
- EfficientNetB3 + BiGRU + MultiHeadAttention
"""
    )

    with gr.Row():

        image_input = gr.Image(
            type="pil",
            label="Upload Food Image"
        )

    predict_btn = gr.Button(
        "🚀 Generate Captions"
    )

    with gr.Row():

        output1 = gr.Markdown()
        output2 = gr.Markdown()
        output3 = gr.Markdown()

    predict_btn.click(
        fn=run_all_models,
        inputs=image_input,
        outputs=[
            output1,
            output2,
            output3
        ]
    )


# ======================================================
# 11. LAUNCH
# ======================================================

demo.launch(debug=True)


2026-05-12 20:17:44.345835: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778617064.536439      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778617064.591072      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778617065.070440      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778617065.070481      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778617065.070484      57 computation_placer.cc:177] computation placer alr

DATASET LOADED ✅
LOADING EfficientNetB0 + BiLSTM...


I0000 00:00:1778617089.790404      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


LOADING ResNet50 + Attention BiLSTM...


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 48 variables whereas the saved optimizer has 52 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


...

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 68 variables whereas the saved optimizer has 72 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



ALL MODELS LOADED ✅
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
CNN BACKBONES READY ✅


/tmp/ipykernel_57/1093635273.py:501: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://c71fe08e9309669223.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


I0000 00:00:1778617182.221141     127 service.cc:152] XLA service 0x58b2ab20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778617182.221207     127 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1778617183.092222     127 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1778617188.855637     127 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
